# Task 3 — Usage MixUp + SAM: folds 0 and 4

This follows the main EDA's saved family folds and the approved expanded v2 dataset.
Each fold starts with **fresh random model weights**. No Gender weights or earlier Usage checkpoint is loaded.

This is one small trial for the rare-class problem. It still predicts all **nine Usage classes**.

| Setting | Fixed value |
|---|---|
| Data | Teacher images + 687 admitted Usage images |
| Model | SmallCNN, trained from scratch |
| Validation folds | **0 and 4 only** |
| Training rows | The other four saved development folds |
| Epochs / checkpoint | 30 / final epoch |
| MixUp | alpha 0.2, every training batch |
| SAM | rho 0.05 over AdamW |
| Class weights | Recomputed from each training fold, beta 0.999, cap 5 |
| Other controls | E8 translation, batch 128, seed 2753, existing cosine schedule |

The completed v2 E8 runs are score references only. Success means better predictions on the
same teacher validation images. A smaller training gap alone is not enough.


## 1. Load the separate training bundle

In **VS Code**, select a fresh **Colab GPU kernel**. Keep these files in `MyDrive/MLA2/data/`:

- `usage_mixup_sam_training.zip`
- Your existing `task3-data.zip`

Choose **Run All**. Results go to `MyDrive/MLA2/task3_usage_mixup_sam/`.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import tempfile
import zipfile

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT), force_remount=False)
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive/MLA2"
BUNDLE_ZIP = DRIVE_PROJECT / "data/usage_mixup_sam_training.zip"
TEACHER_ZIP = DRIVE_PROJECT / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT / "task3_usage_mixup_sam"


def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()


def safe_members(archive, prefix=None):
    members = archive.infolist()
    names = [member.filename for member in members]
    if len(names) != len(set(names)):
        raise ValueError("Archive contains duplicate file names.")
    for member in members:
        name = member.filename
        if Path(name).is_absolute() or ".." in Path(name).parts or "\\" in name:
            raise ValueError(f"Unsafe archive path: {name}")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError(f"Archive symlink is not allowed: {name}")
    return [m for m in members if prefix is None or m.filename.startswith(prefix)]


if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(f"Upload the new training ZIP here first: {BUNDLE_ZIP}")
bundle_digest = sha256_file(BUNDLE_ZIP)
LOCAL_BUNDLE = Path("/content") / f"usage-mixup-sam-{bundle_digest[:12]}.zip"
if not LOCAL_BUNDLE.is_file() or sha256_file(LOCAL_BUNDLE) != bundle_digest:
    partial = LOCAL_BUNDLE.with_suffix(".zip.partial")
    shutil.copyfile(BUNDLE_ZIP, partial)
    if sha256_file(partial) != bundle_digest:
        raise RuntimeError("Training ZIP copy is incomplete. Run this cell again.")
    partial.replace(LOCAL_BUNDLE)
REPO_DIR = Path("/content") / f"MLA2-usage-mixup-sam-{bundle_digest[:12]}"
with zipfile.ZipFile(LOCAL_BUNDLE) as archive:
    members = safe_members(archive)
    manifest = json.loads(archive.read("training_bundle_manifest.json"))
    if manifest["dataset"]["name"] != "teacher_plus_rare_usage_v2_20260906":
        raise ValueError("This notebook requires the new v2 training bundle.")
    if manifest["recipe"]["name"] != "usage_expanded_v2_mixup_sam":
        raise ValueError("This notebook needs the Usage MixUp + SAM bundle.")
    expected = manifest["files"]
    actual = {m.filename for m in members if not m.is_dir()}
    if actual != set(expected) | {"training_bundle_manifest.json"}:
        raise ValueError("The training archive inventory differs from its manifest.")
    for name, digest in expected.items():
        if hashlib.sha256(archive.read(name)).hexdigest() != digest:
            raise ValueError(f"Training archive file changed: {name}")
    if not REPO_DIR.exists():
        with tempfile.TemporaryDirectory(prefix="usage-sam-unpack-", dir="/content") as temporary:
            staging = Path(temporary) / "project"
            staging.mkdir()
            archive.extractall(staging)
            for name, digest in expected.items():
                if sha256_file(staging / name) != digest:
                    raise RuntimeError(f"Extracted bundle file differs: {name}")
            staging.rename(REPO_DIR)
for name, digest in expected.items():
    if not (REPO_DIR / name).is_file() or sha256_file(REPO_DIR / name) != digest:
        raise RuntimeError(f"Local bundle file differs: {name}. Use a fresh runtime.")

if "fashion.config" in sys.modules:
    loaded_root = Path(sys.modules["fashion.config"].ROOT)
    if loaded_root != REPO_DIR:
        raise RuntimeError("Another notebook is loaded. Restart the session, then Run All.")
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
LOCAL_REGISTRY = REPO_DIR / "screen_registry/results/runs.csv"
REFERENCE_ROOT = REPO_DIR / "reference/usage_mixup_sam"
print("Fresh Usage MixUp + SAM code and data ready:", REPO_DIR)
print("Bundle SHA-256:", bundle_digest)


## 2. Put teacher images on local disk

Training reads local images, while saved results go to Drive.


In [ ]:
if not TEACHER_ZIP.is_file():
    raise FileNotFoundError(f"Existing teacher archive is missing: {TEACHER_ZIP}")
LOCAL_TEACHER_ZIP = Path("/content/task3-data.zip")
if (
    not LOCAL_TEACHER_ZIP.is_file()
    or LOCAL_TEACHER_ZIP.stat().st_size != TEACHER_ZIP.stat().st_size
):
    partial = LOCAL_TEACHER_ZIP.with_suffix(".zip.partial")
    shutil.copyfile(TEACHER_ZIP, partial)
    if partial.stat().st_size != TEACHER_ZIP.stat().st_size:
        raise RuntimeError("Teacher ZIP copy is incomplete. Run this cell again.")
    partial.replace(LOCAL_TEACHER_ZIP)
with zipfile.ZipFile(LOCAL_TEACHER_ZIP) as archive:
    members = safe_members(archive, prefix="data/raw/teacher/")
    if not members:
        raise ValueError("Teacher archive must contain data/raw/teacher/.")
    for member in members:
        destination = REPO_DIR / member.filename
        if member.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        elif not destination.is_file() or destination.stat().st_size != member.file_size:
            archive.extract(member, REPO_DIR)
print("Teacher images are on local disk.")


## 3. Check the two saved folds

No new split is made. All split files, image hashes and source groups are checked before fitting.


In [ ]:
import pandas as pd
import torch
from IPython.display import Image as DisplayImage, display
from fashion.train.task3_usage_expanded_v2 import validate_dataset
from fashion.train.task3_usage_mixup_sam import (
    FOLDS,
    run_usage_mixup_sam,
    save_learning_curves,
    screen_spec,
    usage_registry_path,
)

if not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU runtime, then Run All.")
splits, contract = validate_dataset(root=REPO_DIR, check_images=False)
BASELINE_DIR = REFERENCE_ROOT / "baseline"
BASELINE_REGISTRY = REFERENCE_ROOT / "baseline_runs.csv"
DRIVE_REGISTRY = usage_registry_path(DRIVE_TASK_DIR)
assert FOLDS == (0, 4)
print("GPU:", torch.cuda.get_device_name(0))
print("Fresh model weights. Training only folds:", FOLDS)
print("Dataset:", contract["name"])
print("Recipe:", screen_spec().name)
print("Results:", DRIVE_TASK_DIR)
counts = pd.DataFrame(contract["folds"])
display(counts.loc[counts.fold.isin(FOLDS), [
    "fold", "training_rows", "validation_rows",
    "teacher_validation_rows", "new_added_validation_rows",
]])


## 4. Train two fresh models

Both folds start from random weights and stop at epoch 30. The runner rejects any other fold list.
SAM uses the same mixed batch for its two gradient passes and makes one AdamW update per batch.
BatchNorm keeps one update to its running statistics.

A completed fold is reused only after its saved files and training records pass verification.
An interrupted fold starts a fresh run; it does not load partial model weights.


In [ ]:
result = run_usage_mixup_sam(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    baseline_directory=BASELINE_DIR,
    baseline_registry_path=BASELINE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
    folds=(0, 4),
)
print("Saved teacher comparison:", result["comparison_path"])


## 5. Compare the same teacher images

The primary score is macro-F1 on teacher validation images from folds 0 and 4.
Rare-class recall shows the share found correctly. External-source scores are separate diagnostics.

This trial changes MixUp and SAM together. It cannot tell us their separate effects.
This is a development screen, and earlier holdout/test checks already exist.
It does not start the other three folds or pick a final model.


In [ ]:
summary = result["comparison"]
teacher = summary["sources"]["teacher"]
baseline = summary["baseline_sources"]["teacher"]
display(pd.DataFrame([
    {"Model": "Expanded v2 E8", "Teacher macro-F1": baseline["metrics"]["macro_f1"]},
    {"Model": "Fresh MixUp + SAM", "Teacher macro-F1": teacher["metrics"]["macro_f1"]},
]))
print("Same teacher validation images:", teacher["rows"])
print("Teacher macro-F1 change:", summary["teacher_macro_f1_change"])
display(pd.DataFrame(summary["fold_comparison"]))
display(pd.DataFrame(summary["teacher_per_class"]))
display(pd.DataFrame([
    {"Source": name, "Images": scope["rows"],
     "Macro-F1": scope["metrics"]["macro_f1"] if scope["metrics"] else None}
    for name, scope in summary["sources"].items()
]))
print("Result files:", Path(result["comparison_path"]).parent)
print("Run log:", result["registry_path"])


## 6. Check learning curves

These curves show **combined** validation scores and class-weighted losses.
Mixed training images have no ordinary training F1. Use the clean teacher training/validation
table above for the final gap. The curves do not select a different checkpoint.


In [ ]:
curve_path = save_learning_curves(
    result,
    path=DRIVE_TASK_DIR / "results/figures/task3/usage_mixup_sam_learning_curves.png",
)
display(DisplayImage(filename=str(curve_path)))
print("Saved curves:", curve_path)
